<a href="https://colab.research.google.com/github/anu5hkaa/AI-War-News-Analyser-System/blob/main/War%20news%20analzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import numpy as np

In [ ]:
api_key='e2a5d3214eeb40e297e0ab4e64e4b510'

In [ ]:

url = "https://newsapi.org/v2/everything"

war_keywords = [
    "war", "conflict", "military", "airstrike", "missile",
    "bomb", "attack", "invasion", "army", "troops",
    "defense", "clash", "border", "violence", "strike"
]
queries = [
    "war",
    "military conflict",
    "airstrike attack",
    "border clash",
    "missile strike",
    "army operation",
    "defense military",
    "troops conflict"
]

all_articles = []
news_list = []
for query in queries:
    for page in range(1, 6):   # 5 pages × 20 = 100 per query

        params = {
            "q": query,
            "language": "en",
            "sortBy": "publishedAt",
            "pageSize": 20,   # free API limit per request
            "page": page,
            "apiKey": api_key
        }

        response = requests.get(url, params=params)
        data = response.json()

        articles = data.get("articles", [])
        all_articles.extend(articles)

for article in all_articles:
    news = {
        "title": article.get("title", ""),
        "content": article.get("description", ""),
        "source": article.get("source", {}).get("name", ""),
        "url": article.get("url", ""),
        "date": article.get("publishedAt", "")
    }
    news_list.append(news)

In [ ]:

df = pd.DataFrame(news_list)

In [ ]:
df.head(10)

,title,content,source,url,date
0,Air Canada to suspend flights to JFK for month...,It’s the latest move in an industry struggling...,Boston Herald,https://www.bostonherald.com/2026/04/17/air-ca...,2026-04-17T17:18:25Z
1,"Former UDFA Arian Foster reflects on Saints, T...",Before he became the Texans' all-time rushing ...,USA Today,https://saintswire.usatoday.com/story/sports/n...,2026-04-17T17:17:48Z
2,"[US] Hotfixes: April 17, 2026",[Blue Topic] Here you'll find a list of hotfix...,Bluetracker.gg,https://www.bluetracker.gg/wow/topic/us-en/242...,2026-04-17T17:17:14Z
3,"[EU] Hotfixes: April 17, 2026",[Blue Topic] Here you'll find a list of hotfix...,Bluetracker.gg,https://www.bluetracker.gg/wow/topic/eu-en/242...,2026-04-17T17:17:14Z
4,Is the men’s World Cup hindering the women’ 20...,When U.S. Soccer president Cindy Parlow Cone a...,Yahoo Entertainment,https://sports.yahoo.com/articles/men-world-cu...,2026-04-17T17:16:05Z
5,MLB franchise closing in on record-breaking $3...,The San Diego Padres are reportedly on the bri...,Alloutsoccer.com,https://www.alloutsoccer.com/news/mlb-padres-s...,2026-04-17T17:16:02Z
6,Redshift,"""Rehearsing for humanity’s future on Mars.""",Longreads.com,http://longreads.com/2026/04/17/mars-desert-re...,2026-04-17T17:14:01Z
7,The Complete Gears of War Timeline (So Far),,IGN,https://es.ign.com/gallery/168082/the-complete...,2026-04-17T17:13:41Z
8,Trump Taunts Tucker Carlson With Recent Poll: ...,The MAGA leader also insulted conservative pun...,HuffPost,https://www.huffpost.com/entry/trump-taunts-tu...,2026-04-17T17:13:31Z
9,"Stock market today: Dow climbs 1,000 points, S...","Stock market today: Dow climbs 1,000 points, S...",Slashdot.org,https://slashdot.org/firehose.pl?op=view&amp;i...,2026-04-17T17:12:51Z


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 752 entries, 0 to 751
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    752 non-null    object
 1   content  749 non-null    object
 2   source   752 non-null    object
 3   url      752 non-null    object
 4   date     752 non-null    object
dtypes: object(5)
memory usage: 29.5+ KB


In [ ]:
df['content'][4]

'When U.S. Soccer president Cindy Parlow Cone announced in October her federation’s bid to co-host the 2031 FIFA Women’s World Cup, the excitement in the room was palpable. “As the only bidders, I admit, I like our chances,” she said, with a smile, to the stan…'

In [ ]:
len((df))

752

In [ ]:
df = df[~df["title"].str.lower().str.contains("movie|film|trailer|review|box office")]

In [ ]:
len(df)

743

In [ ]:
from transformers import pipeline

# Load once (don’t put inside loop)
classifier = pipeline("zero-shot-classification")

def is_war_news(text):
    labels = ["war or military news", "not related"]

    result = classifier(text, labels)

    return result["labels"][0] == "war or military news"

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [ ]:
# 🔥 FINAL AI FILTER
df = df[df["title"].apply(is_war_news)]

In [ ]:
print(len(df))


620


In [ ]:
!pip install sentence-transformers faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # This is a good, efficient model

df["title"] = df["title"].fillna('').str.lower()
df["content"] = df["content"].fillna('').str.lower()
df["source"] = df["source"].astype(str).str.lower()
# Prepare your text data: combine title and content or just use one
df['combined_text'] = df['title'] + " " + df['content']

# Convert the combined text into embeddings
embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
embeddings.shape


(620, 384)

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings)

# IMPORTANT → normalize for cosine similarity
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)  # cosine similarity
index.add(embeddings)

In [ ]:
query = "pakistan and india at war"

query_embedding = model.encode([query])
query_embedding = np.array(query_embedding)

faiss.normalize_L2(query_embedding)

In [ ]:
D, I = index.search(query_embedding, k=5)

In [ ]:
for i in I[0]:
    print(df.iloc[i]["title"])
    print(df.iloc[i]["source"])
    print(df.iloc[i]['url'])
    print("-----")

pakistan boosts mediation efforts as us, iran weigh longer truce
financial post
https://financialpost.com/pmn/business-pmn/pakistan-boosts-mediation-efforts-as-us-iran-weigh-longer-truce
-----
pakistan boosts mediation efforts as us, iran weigh longer truce
financial post
https://financialpost.com/pmn/business-pmn/pakistan-boosts-mediation-efforts-as-us-iran-weigh-longer-truce
-----
eid pause over: pakistan, afghanistan trade fire again; 2 civilians killed, several injured
the times of india
https://timesofindia.indiatimes.com/world/south-asia/eid-pause-over-pakistan-afghanistan-trade-fire-again-2-killed-several-injured/articleshow/129829631.cms
-----
leaked documents reveal details of the secret saudi arabia-pakistan mutual defense pact
globalresearch.ca
https://www.globalresearch.ca/secret-saudi-arabia-pakistan-mutual-defense-pact/5922705
-----
leaked documents reveal details of the secret saudi arabia-pakistan mutual defense pact
globalresearch.ca
https://www.globalresearch.ca/secre

In [ ]:
!pip install transformers



In [ ]:
!pip install feedparser

In [ ]:
# =========================
# IMPORTS
# =========================
import numpy as np
import pandas as pd
import faiss
import feedparser
import requests
from sentence_transformers import SentenceTransformer, CrossEncoder


# =========================
# LOAD MODELS
# =========================
model = SentenceTransformer('all-MiniLM-L6-v2')
cross_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


# =========================
# PREPARE DATA
# =========================
df['content'] = df['content'].fillna('').astype(str).str.lower()
df['title'] = df['title'].fillna('').astype(str).str.lower()
df['date'] = df['date'].astype(str)

df['full_text'] = df['title'] + " " + df['content']


# =========================
# EMBEDDINGS + INDEX
# =========================
embeddings = model.encode(df['full_text'].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings)
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)


# =========================
# WAR KEYWORDS
# =========================
war_keywords = [
    "war", "conflict", "attack", "military", "missile",
    "battle", "army", "troops", "airstrike", "bomb",
    "border", "clash", "violence", "defense", "ceasefire"
]


# =========================
# QUERY EXPANSION
# =========================
def expand_query(query):
    return query + " war conflict military attack"


# =========================
# FINAL SMART FILTER (🔥 KEY LOGIC)
# =========================
def smart_filter(results, query):
    words = [w for w in query.split() if len(w) > 2]

    filtered = []

    for item in results:
        text = (item["title"] + " " + item["content"]).lower()

        # must be war-related
        if not any(k in text for k in war_keywords):
            continue

        if len(words) == 1:
            # single word → flexible
            if words[0] in text:
                filtered.append(item)
        else:
            # multiple words → strict (ALL must match)
            if all(word in text for word in words):
                filtered.append(item)

    return filtered


# =========================
# CLEAN LINKS
# =========================
def clean_google_link(link):
    try:
        response = requests.get(link, allow_redirects=True, timeout=5)
        return response.url
    except:
        return link


# =========================
# FETCH RSS
# =========================
def fetch_rss_news(query):

    query_formatted = query.replace(" ", "+")

    rss_urls = [
        f"https://news.google.com/rss/search?q={query_formatted}+war",
        f"https://news.google.com/rss/search?q={query_formatted}+conflict",
        f"https://news.google.com/rss/search?q={query_formatted}+military",

        "https://www.defensenews.com/arc/outboundfeeds/rss/?outputType=xml",
        "https://www.militarytimes.com/arc/outboundfeeds/rss/",
        "https://www.armytimes.com/arc/outboundfeeds/rss/"
    ]

    rss_data = []

    for url in rss_urls:
        feed = feedparser.parse(url)

        for entry in feed.entries:
            text = entry.title.lower()

            if any(k in text for k in war_keywords):
                rss_data.append({
                    "title": entry.title.lower(),
                    "content": entry.title,
                    "url": clean_google_link(entry.link),
                    "date": entry.published if "published" in entry else "",
                    "full_text": entry.title.lower()
                })

    # remove duplicates
    seen = set()
    unique = []

    for item in rss_data:
        if item["url"] not in seen:
            unique.append(item)
            seen.add(item["url"])

    return unique[:40]


# =========================
# MAIN FUNCTION
# =========================
def get_top_matches(query):

    original_query = query.lower().strip()
    expanded_query = expand_query(original_query)

    print("Searching...")

    # =========================
    # FAISS SEARCH
    # =========================
    query_embedding = model.encode([expanded_query])
    query_embedding = np.array(query_embedding)
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k=20)

    api_results = []

    for i in I[0]:
        if i < len(df):
            api_results.append({
                "title": str(df.iloc[i]['title']),
                "content": str(df.iloc[i]['content'])[:1000],
                "url": df.iloc[i]['url'],
                "date": df.iloc[i]['date']
            })

    # =========================
    # RSS DATA
    # =========================
    rss_results = fetch_rss_news(original_query)

    # =========================
    # RANKING
    # =========================
    api_pairs = [(expanded_query, item["title"] + " " + item["content"]) for item in api_results]
    api_scores = cross_model.predict(api_pairs) if api_pairs else []
    api_ranked = sorted(zip(api_scores, api_results), reverse=True)

    rss_pairs = [(expanded_query, item["title"] + " " + item["content"]) for item in rss_results]
    rss_scores = cross_model.predict(rss_pairs) if rss_pairs else []
    rss_ranked = sorted(zip(rss_scores, rss_results), reverse=True)

    combined = [item for _, item in (api_ranked[:5] + rss_ranked[:5])]

    # =========================
    # FINAL FILTER
    # =========================
    filtered = smart_filter(combined, original_query)

    if not filtered:
        print("Not Found")
        return []

    final_results = filtered[:10]

    # =========================
    # CLEAN OUTPUT
    # =========================
    clean_results = []

    for item in final_results:
        try:
            parsed = pd.to_datetime(item["date"], errors='coerce')
            date_str = parsed.strftime("%Y-%m-%d") if not pd.isna(parsed) else str(item["date"])
        except:
            date_str = str(item["date"])

        clean_results.append({
            "title": item["title"],
            "content": item["content"],
            "url": item["url"],
            "date": date_str
        })

    # =========================
    # SAFE LLM SUMMARY
    # =========================
    if len(clean_results) >= 2:
        prompt = create_prompt(original_query, clean_results)
        summary = get_llm_response(prompt)

        print("\nSummary:\n")
        print(summary)

    print("\nSources:\n")

    for i, news in enumerate(clean_results):
        print(f"{i+1}. {news['title']}")
        print(news["url"])
        print("Date:", news["date"])
        print()









Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
!pip install Groq

In [ ]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"]=getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [ ]:
from groq import Groq
client=Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
def create_prompt(query, clean_results):

    prompt = f"""
You are a war news analyst.

User Query: {query}

Below are some news articles:

"""

    for i, news in enumerate(clean_results, 1):
        prompt += f"""
Article {i}:
Title: {news['title']}
Content: {news['content']}
Date: {news['date']}
"""

    prompt += """
Task:
- Summarize what is happening
- Explain in simple simple words
-Key insights:
- Keep answer short
"""

    return prompt

In [ ]:
def get_llm_response(prompt):

    print("LLM running...")

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    print("LLM done!")

    return response.choices[0].message.content

In [ ]:
get_top_matches("anushka ")

Searching...
LLM running...
LLM done!

Summary:

**What’s happening**  
- Several Indian news outlets are publishing the same story about actress Anushka Sharma.  
- She is recalling memories from 1999 when her father, an Indian Army officer, fought in the Kargil War.  
- As an 11‑year‑old she says she was scared seeing her mother worry and admits she was “insensitive” to her dad’s duties at the time.  
- The pieces are appearing now (May 2025) as India‑Pakistan tensions flare again, so the media are linking the personal recollection to the current security climate.

**Simple explanation**  
Anushka Sharma talked about how, when she was a child, her dad was away fighting in the Kargil conflict. She felt afraid and later realized she didn’t understand what her dad was doing. The story is being retold now because India and Pakistan are tense again, and the press is using her personal memory to illustrate the human side of war.

**Key insights**  
- **Human angle:** Personal family storie